# 5. Evaluation

This section assesses whether the modelling results meet the business objectives defined in Business Understanding, reviews the overall process, and determines next steps.

## 5.1 Evaluate Results

### Assessment Against Business Success Criteria

The business success criteria defined at the start of the project were: at least three distinct customer segments with clear and differentiated behavioural profiles and  actionable recommendations covering retention, re-engagement, and channel prioritisation.

The first criterion is met. K=4 produces four segments with clearly different profiles across recency, revenue, and tenure.

The second criterion is met. Actionable recommendations are documented in section 4.5 of the Modeling notebook and are summarised in section 5.1.1 below.


### Assessment Against Data Mining Success Criteria

The silhouette score target of 0.30 or above was met, with the final model achieving a score of 0.462. This is a strong result for a real-world behavioural dataset of this size, where most customers have only one recorded stay and the inherent variation in behaviour is limited.

All clusters differ significantly on all three modelling variables, exceeding the requirement for differentiation on at least two dimensions.

All four segments were named in business terms and are interpretable without statistical expertise.

The only borderline result is Cluster 3 at 5.9 percent, as noted above. All other criteria are fully met.


### 5.1.1 Final Segment Summary

The four segments produced by the K=4 K-Means model are:

#### Cluster 1: Active Guests (31.2%)

Last stay within the past 369 days. Average revenue €434.50. Mean tenure 8.34 days. These are the most recently active customers. They include a mix of nationalities (FRA, PRT, DEU, GBR in similar proportions) and are predominantly reached through Travel Agent/Operator. They represent the core retention target: recent enough to be reachable and engaged enough to be worth investing in.

**Marketing action**: retention. Personalised offers timed ahead of the next expected travel window. Invite into loyalty programme if one exists.

#### Cluster 2: Lapsed Guests (32.2%)

Last stay 355–720 days ago. Average revenue €403.28. Mean tenure 2.96 days. Predominantly one-time visitors who stayed once and have not returned. Similar in revenue and channel mix to Active Guests but further from their last stay. PRT customers are relatively more present here.

**Marketing action**: re-engagement. Winback campaigns through Travel Agent/Operator channels. Seasonal promotions with price incentives.

#### Cluster 0: Dormant Guests (30.6%)

Last stay 714–1,104 days ago. Lowest average revenue €387.58. Near-zero tenure. The least engaged group: no relationship with the hotel in nearly three years. PRT customers are most over-represented here (19.9%), suggesting domestic customers are more likely to lapse into dormancy.

**Marketing action**: low-investment reactivation. Broad seasonal campaigns. Lower expected response rate than Lapsed Guests.

#### Cluster 3: High-Value Loyal Guests (5.9%)

Average revenue €1,588.15, nearly four times the other segments. Mean tenure 33.59 days, highest bookings checked in (1.37), highest room nights (6.00). These customers have a demonstrably longer relationship with the hotel. They are more likely to book directly (21.3% Direct vs 15–18% in other segments) and are more likely to request quiet rooms (13.3%). PRT customers are under-represented (9.3%).

**Marketing action**: premium loyalty. Priority treatment, exclusive offers, dedicated service. Maybe a structured loyalty programme targeted at this segment and a direct channel cultivation to reduce reliance on Travel Agent/Operator.

### 5.1.2 Additional Findings

Beyond the business objectives, the data mining exercise surfaced several findings that were not explicitly requested but are relevant to A:

- **Ghost customers (23.8% of the database)** are a significant data quality issue. Nearly one in four customer records has never produced any stay or revenue. These profiles inflate the apparent size of the customer base and should be excluded from any reporting or campaign targeting.

- **Split profiles**: 4,093 rows sharing both NameHash and DocIDHash with different behavioural records represent the same customer with a fragmented history in the PMS. These were consolidated before clustering.

- **Market segment and distribution channel do not differentiate segments**, except for Cluster 3. For 94% of customers (Clusters 0–2), channel and segment composition is nearly identical. The hotel's existing origin-based segmentation therefore provides no useful differentiation beyond identifying the High-Value Loyal group.

- **Portuguese customers are disproportionately dormant.** PRT represents 19.9% of Cluster 0 (Dormant) but only 9.3% of Cluster 3 (High-Value Loyal). Domestic customers appear to be high-frequency but low-loyalty visitors.

## 5.2 Review Process

### Process Overview

The project followed the full CRISP-DM cycle: Business Understanding → Data Understanding → Data Preparation → Modeling → Evaluation.

### What worked well

- **Ghost customer removal** was the single most impactful preparation decision. Removing 19,920 records with no activity prevented a dominant zero-cluster from masking real segments.

- **Feature engineering produced a genuine improvement.** The `Tenure` variable (DaysSinceFirstStay − DaysSinceLastStay) extracted an independent loyalty signal that neither raw variable alone captured. Without it, the K=4 solution would not have identified Cluster 3.

- **Exhaustive feature search.** Testing all subsets of 3–6 variables with multiple scalers and K values ensured that the final model inputs were chosen on evidence, not assumption. This is what produced a silhouette of 0.462 rather than the initial 0.087.

- **Split profile merging** restored the full behavioural history of customers split across multiple PMS records, directly improving the quality of the Cluster 3 signal.

### What could be improved

- **Cluster 3 stability.** The silhouette plot shows that Cluster 3 has some negative silhouette values, some customers near its boundary are geometrically closer to another cluster.

- **DaysSinceLastStay as the primary separator.** The three main clusters (Active, Lapsed, Dormant) are separated almost entirely by recency. Revenue and tenure add secondary differentiation. This reflects the structure of the data, most customers stayed once, but limits the quality of the segmentation.

## 5.3 Determine Next Steps

### Recommended Actions

**1. Deploy the segmentation.** The K=4 model is ready for use. The cluster labels should be attached to the customer database in the PMS and made available to the marketing department for campaign targeting.

**2. Prioritise Cluster 3 first.** High-Value Loyal Guests generate disproportionate revenue and already show a preference for direct booking. A dedicated loyalty programme for this group is the highest-ROI action.

**3. Re-engage Cluster 2 before Cluster 0.** Lapsed Guests are closer to their last stay and therefore more likely to respond to winback campaigns. Dormant Guests should receive lower-cost broad campaigns.

**4. Audit the PMS data entry process.** The split profiles and ghost customers found in the dataset indicate systematic data quality issues. Addressing these at source will improve future segmentation quality.

### Decision

The project proceeds to deployment.

# 6. Deployment

This section documents how the project's results should be deployed, monitored, and maintained.

## 6.1 Deployment Plan

### Deployable Results

The project produces two deployable outputs: the customer segmentation table (df_profile with Cluster and Segment columns attached, covering 60,903 active adult customers) and the fitted model artefacts (MinMaxScaler and KMeans saved as .pkl files for scoring new customers without rerunning the pipeline).

### Deployment Steps

**Step 1: Load the segmentation into the PMS or CRM.**

Import the segmentation table into the hotel's system, joining on customer ID. Add the four segment labels as a new field on each customer record.

**Step 2: Enable segment-based targeting in campaign tools.**

Configure each of the four segments as a separate audience in the email marketing or CRM tool.

**Step 3: Score new customers as they are created.**

Apply the saved scaler and KMeans model to each new customer's DaysSinceLastStay, TotalRevenue, and Tenure values. The predicted cluster maps directly to a segment label.

**Step 4: Communicate the segmentation to the marketing team.**

Share the segment descriptions in section 6.3 with A and the campaign team.

### Segment Action Plan

- High-Value Loyal Guests (5.9%) are the top priority: dedicated loyalty programme, exclusive offers, direct channel cultivation. 

- Active Guests (31.2%) are the core short-term retention target: personalised offers and loyalty programme invitation. 

- Lapsed Guests (32.2%) are the medium-term re-engagement target: winback campaigns with seasonal promotions through Travel Agent channels. 

- Dormant Guests (30.6%) are a low-investment long-term target: one broad annual reactivation communication with no expectation of high response rates.

## 6.2 Monitoring and Maintenance

### Monitoring Approach

Segment size should be tracked quarterly. A shift of more than 5 percentage points in any segment signals that the model no longer reflects the current customer base. Mean TotalRevenue and DaysSinceLastStay per segment should also be monitored.

### When to Refresh the Model

Rerun the model when more than 12 months have passed since the last segmentation, when the active customer base grows by more than 20 percent, or when a significant strategic change occurs such as launching a loyalty programme or entering a new market.

### Limitations on Use

Dormant Guests should still receive low-frequency communications so that reactivation rates can be tracked over time.

## 6.3 Final Report

### Project Summary

Hotel H's customer database was segmented into four behavioural groups using K-Means clustering on DaysSinceLastStay, TotalRevenue, and Tenure, covering 60,903 active adult customers. The model achieved a silhouette score of 0.462.

### Segment Descriptions


**High-Value Loyal Guests (5.9%, approximately 3,600 customers)**

Average spend of €1,588, nearly four times any other segment. They return repeatedly and are more likely to book directly. This group should receive dedicated loyalty treatment and priority service.


**Active Guests (31.2%, approximately 19,000 customers)**

Stayed within the last year, average spend €434. The easiest group to retain. Communicate ahead of the travel season and invite into the loyalty programme.

**Lapsed Guests (32.2%, approximately 19,600 customers)**

Stayed one to two years ago and have not returned. They are reachable but need an incentive. Targeted seasonal promotions through their original booking channel are most likely to work.


**Dormant Guests (30.6%, approximately 18,700 customers)**

No stay in nearly three years. Reactivation cost is high relative to expected return. A low-frequency broad campaign is appropriate, as stated above.

### Deviations from Original Plan

Three segments were initially assumed to be sufficient. The data revealed a fourth commercially important segment identifiable only after engineering the Tenure variable, leading to the selection of K=4. Ghost customers (23.8 percent of records) required removal before any clustering could produce meaningful results. PCA was tested but not applied: with three modelling variables it added no benefit over MinMaxScaler alone.

## 6.4 Review Project

### What Went Well

Exhaustive feature subset testing increased the silhouette score from 0.087 to 0.462. The Tenure feature, derived from the difference between first and last stay dates, was not in the original variable set but directly enabled the identification of the High-Value Loyal segment. All preparation decisions were grounded in business reasoning.

### What Could Be Improved

The hierarchical dendrogram relied on a single 2,000-customer sample, multiple samples would have produced a better K guidance. The three main clusters are separated primarily by recency, reflecting a dataset where most customers stayed once. Transaction-level data would produce better behavioural variation and sharper cluster boundaries in the future.